In [2]:
import data_collection as data
import nflreadpy as nfl
import pandas as pd
import numpy as np


In [59]:
import pandas as pd
from typing import Optional


def append_week_lines_to_historic(
    week_lines_csv: str = "vegas/week_11_lines.csv",
    historic_csv: str = "data/historic_lines.csv",
    teams_csv: str = "data/nfl_teams.csv",
    schedule_season: int = 2025,
    schedule_week: int = 11,
    bookmaker_preference: Optional[str] = "DraftKings"
) -> int:
    """Append weekly spread lines into historic_lines.csv, preserving schema.

    - Reads week-level lines in the format of week_2_lines.csv (two rows/team per game).
    - Maps full team names to IDs matching historic_lines.csv via nfl_teams.csv.
    - Aggregates to one row per game: picks the favorite (negative point), keeps total.
    - Appends new rows to historic_lines.csv with the same columns and blank index header.

    Returns
    -------
    int
        Number of rows appended (deduped against existing season/week/home/away).
    """
    # Load team mapping (full name -> team_id used in historic file)
    teams_df = pd.read_csv(teams_csv)
    name_to_id = dict(zip(teams_df["team_name"].astype(str), teams_df["team_id"].astype(str)))

    # Load week lines and filter to spreads (and bookmaker if provided)
    week_df = pd.read_csv(week_lines_csv)
    if "market" in week_df.columns:
        week_df = week_df[week_df["market"].str.lower() == "spreads"].copy()
    if bookmaker_preference and "bookmaker" in week_df.columns:
        week_df = week_df[week_df["bookmaker"].astype(str) == bookmaker_preference].copy()

    # Normalize numeric fields
    if "point" in week_df.columns:
        week_df["point"] = pd.to_numeric(week_df["point"], errors="coerce")
    # 'over/under' has a slash in the name; keep safe access
    ou_col = "over/under" if "over/under" in week_df.columns else (
        "over_under" if "over_under" in week_df.columns else None
    )
    if ou_col is not None:
        week_df[ou_col] = pd.to_numeric(week_df[ou_col], errors="coerce")

    # Group to one row per game
    required_cols = {"game_id", "home_team", "away_team", "label"}
    missing = [c for c in required_cols if c not in week_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {week_lines_csv}: {missing}")

    records = []
    for game_id, grp in week_df.groupby("game_id", sort=False):
        home_team_name = str(grp["home_team"].iloc[0])
        away_team_name = str(grp["away_team"].iloc[0])

        # Determine favorite: row with the most negative spread (minimum point)
        grp_nonnull = grp.dropna(subset=["point"]) if "point" in grp.columns else grp.copy()
        if grp_nonnull.empty:
            # If we can't determine a favorite, skip this game
            continue
        fav_idx = grp_nonnull["point"].idxmin()
        fav_team_name = str(grp_nonnull.loc[fav_idx, "label"])  # team name in the bet label
        spread_favorite = float(grp_nonnull.loc[fav_idx, "point"])  # should be negative

        # Over/Under line: take first non-null within the game
        if ou_col is not None:
            ou_series = grp_nonnull[ou_col].dropna()
            over_under_line = float(ou_series.iloc[0]) if not ou_series.empty else None
        else:
            over_under_line = None

        # Map to team IDs used in historic file
        home_id = name_to_id.get(home_team_name)
        away_id = name_to_id.get(away_team_name)
        fav_id = name_to_id.get(fav_team_name)

        if home_id is None or away_id is None or fav_id is None:
            # Try a couple of common aliases
            alias = {
                "LA Rams": "Los Angeles Rams",
                "LA Chargers": "Los Angeles Chargers",
                "LV Raiders": "Las Vegas Raiders",
                "Washington": "Washington Commanders",
            }
            home_id = home_id or name_to_id.get(alias.get(home_team_name, home_team_name))
            away_id = away_id or name_to_id.get(alias.get(away_team_name, away_team_name))
            fav_id = fav_id or name_to_id.get(alias.get(fav_team_name, fav_team_name))

        if home_id is None or away_id is None or fav_id is None:
            raise KeyError(
                f"Missing team_id mapping. home='{home_team_name}'->{home_id}, "
                f"away='{away_team_name}'->{away_id}, favorite='{fav_team_name}'->{fav_id}"
            )

        records.append({
            "schedule_season": int(schedule_season),
            "schedule_week": int(schedule_week),
            "team_home": home_team_name,
            "team_away": away_team_name,
            "team_favorite_id": fav_id,
            "spread_favorite": spread_favorite,
            "over_under_line": over_under_line,
            "schedule_playoff": False,
            "team_home_id": home_id,
            "team_away_id": away_id,
        })

    new_rows_df = pd.DataFrame.from_records(records)
    if new_rows_df.empty:
        return 0

    # Load historic lines with existing index (blank header) and dedupe by key
    historic_df = pd.read_csv(historic_csv, index_col=0, low_memory=False)
    historic_df.index.name = ""  # Ensure blank header on index when saving

    new_rows_df["__key"] = (
        new_rows_df["schedule_season"].astype(str)
        + "|" + new_rows_df["schedule_week"].astype(str)
        + "|" + new_rows_df["team_home"].astype(str)
        + "|" + new_rows_df["team_away"].astype(str)
    )
    hist_keys = set(
        (historic_df["schedule_season"].astype(str)
         + "|" + historic_df["schedule_week"].astype(str)
         + "|" + historic_df["team_home"].astype(str)
         + "|" + historic_df["team_away"].astype(str))
        .values
    )

    new_rows_df = new_rows_df[~new_rows_df["__key"].isin(hist_keys)].drop(columns=["__key"])  # anti-join
    if new_rows_df.empty:
        return 0

    # Assign sequential index values continuing from existing max index
    try:
        start_index = int(pd.to_numeric(pd.Series(historic_df.index)).max())
    except Exception:
        # If index isn't numeric for some reason, fall back to length-1
        start_index = len(historic_df) - 1

    new_index = list(range(start_index + 1, start_index + 1 + len(new_rows_df)))
    new_rows_df.index = new_index
    new_rows_df.index.name = ""  # keep blank index header

    # Concatenate and persist
    out_df = pd.concat([historic_df, new_rows_df], axis=0)
    out_df.index.name = ""
    out_df.to_csv(historic_csv)

    return len(new_rows_df)

# Example usage (uncomment to run):
# appended = append_week_lines_to_historic()
# print(f"Appended {appended} rows to historic_lines.csv")


In [ ]:
#appended = append_week_lines_to_historic()

In [79]:
def get_weekly_scorers(season, week):
    pbp = nfl.load_pbp(seasons=[season]).to_pandas()

    # Filter for regular season, Week 1
    week_pbp = pbp[(pbp['week'] == week)]

    # Keep only touchdown plays
    week_tds = week_pbp[week_pbp['touchdown'] == 1]

    # Count TDs per scorer. Prefer id+name if both available, else fall back to name only
    use_cols = [c for c in ['td_player_id', 'td_player_name'] if c in week_tds.columns]
    if use_cols:
        scorers = (
            week_tds.dropna(subset=use_cols)
            .groupby(use_cols)
            .size()
            .reset_index(name='tds')
        )
        if 'td_player_id' in use_cols:
            scorers = scorers.rename(columns={'td_player_name': 'player', 'td_player_id': 'player_id'})
        else:
            scorers = scorers.rename(columns={'td_player_name': 'player'})
    else:
        # Fallback if td_* columns not present; derive from rusher/receiver
        rush = week_tds.dropna(subset=['rusher_player_id'])[['rusher_player_id', 'rusher_player_name']]
        rec = week_tds.dropna(subset=['receiver_player_id'])[['receiver_player_id', 'receiver_player_name']]
        rush.columns = ['player_id', 'player']
        rec.columns = ['player_id', 'player']
        both = pd.concat([rush, rec], ignore_index=True)
        scorers = both.groupby(['player_id', 'player']).size().reset_index(name='tds')

    return scorers

In [80]:
def get_weekly_results(season,week):       
    scorers = get_weekly_scorers(2025, week)
    
    predictions = pd.read_csv(f'predictions/predictions_week_{week}.csv')

    #Keep only player_id, player_display_name, predicted_touchdown_probability, and model_edge
    predictions = predictions[['player_id', 'player_display_name', 'position','team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]

    #Sort by predicted_touchdown_probability in descending order
    predictions = predictions.sort_values(by='predicted_touchdown_probability', ascending=False)

    # Join predictions with scorers: prefer player_id, else fall back to name
    if 'player_id' in scorers.columns:
        pred_scored = predictions.merge(
            scorers[['player_id', 'tds']], on='player_id', how='inner'
        )
    else:
        pred_scored = predictions.merge(
            scorers[['player', 'tds']], left_on='player_display_name', right_on='player', how='inner'
        )

    return pred_scored


In [81]:
def simulate_betting(df, scorers, use_kelly=False):
    stake = 10.0

    bets = df.copy()

    # Merge to mark hits
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0

    # American odds payout logic
    # profit_if_win = stake * (odds/100) if odds > 0 else stake * (100/abs(odds))
    # profit_if_loss = -stake
    is_plus = bets['price'] > 0
    profit_if_win = stake * (bets['price'] / 100.0)
    profit_if_win = profit_if_win.where(is_plus, stake * (100.0 / bets['price'].abs()))

    bets['profit'] = np.where(bets['hit'], profit_if_win, -stake)

    # Add total return
    bets['return'] = stake + bets['profit']

    # Summary metrics
    num_bets = len(bets)
    hits = int(bets['hit'].sum())
    hit_rate = hits / num_bets if num_bets else 0.0
    total_profit = float(bets['profit'].sum())
    roi = total_profit / (stake * num_bets) if num_bets else 0.0

    summary = {
        'bets': num_bets,
        'hits': hits,
        'hit_rate': round(hit_rate, 3),
        'total_profit': round(total_profit, 2),
        'roi': round(roi, 3)
    }

    display(summary)

    # Show detailed results
    cols = [
        'player_id', 'player_display_name', 'team', 'position', 'price',
        'predicted_touchdown_probability', 'model_edge', 'tds', 'hit', 'profit', 'return'
    ]
    return bets[cols].sort_values(['predicted_touchdown_probability'], ascending=[False]).reset_index(drop=True), summary


In [82]:
def ev_betting(season, week,):
    scorers = get_weekly_results(season, week)
    predictions = pd.read_csv(f'predictions/predictions_week_{week}.csv')
    predictions.drop_duplicates(subset=['player_id'], inplace=True)



    
    

    top_te = predictions[predictions['position'] == 'TE'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    top_wr = predictions[predictions['position'] == 'WR'].sort_values(by='predicted_touchdown_probability', ascending=False).head(5)
    #ev_wr = predictions[predictions['position'] == 'WR'].sort_values(by='model_edge', ascending=False)
    #ev_wr = ev_wr[ev_wr['price'] <= 400].head(5)

    neg = top_wr[top_wr['model_edge'] >= 0]

    ev_te = predictions[predictions['position'] == 'TE']
    ev_te = ev_te[ev_te['model_edge'] >= 0.025] 
    ev_te = ev_te[ev_te['price'] <= 400]

    ev = predictions[predictions['model_edge'] >= 0.05]
    ev = ev[ev['model_edge'] <= 0.10]
    ev = ev[ev['price'] <= 400]
   
    ev_wr = predictions[predictions['position'] == 'WR'].sort_values(by='predicted_touchdown_probability', ascending=False).head(20)
    ev_wr = ev_wr[ev_wr['model_edge'] >= 0]

    top_pick = predictions[predictions['price'] <= 400]
    top_pick = top_pick.sort_values(by='model_edge', ascending=False).head(1)

    threshold = predictions[predictions['predicted_touchdown_probability'] >= 0.43]


    
    return simulate_betting(top_wr, scorers)


In [83]:
def get_total_results():
    total_bets = 0
    total_hits = 0
    total_profit = 0
    for i in range(1, 12):
        df, roi = ev_betting(2025, i)
        #print(df)
        total_bets += roi['bets']
        total_hits += roi['hits']
        total_profit += roi['total_profit']

    print(f"Total bets: {total_bets}")
    print(f"Total hits: {total_hits}")
    print(f"Total profit: {total_profit}")
    print(f"Hit rate: {total_hits / total_bets}")
    print(f"ROI: {total_profit / (total_bets * 10.0)}")

In [84]:
get_total_results()

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': -3.5, 'roi': -0.07}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 13.19, 'roi': 0.264}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 17.5, 'roi': 0.35}

{'bets': 5, 'hits': 5, 'hit_rate': 1.0, 'total_profit': 66.5, 'roi': 1.33}

{'bets': 5, 'hits': 3, 'hit_rate': 0.6, 'total_profit': 15.5, 'roi': 0.31}

...

{'bets': 5, 'hits': 0, 'hit_rate': 0.0, 'total_profit': -50.0, 'roi': -1.0}

{'bets': 5, 'hits': 1, 'hit_rate': 0.2, 'total_profit': -32.0, 'roi': -0.64}

{'bets': 5, 'hits': 4, 'hit_rate': 0.8, 'total_profit': 19.62, 'roi': 0.392}

{'bets': 5, 'hits': 2, 'hit_rate': 0.4, 'total_profit': -6.67, 'roi': -0.133}

Total bets: 55
Total hits: 28
Total profit: 43.17
Hit rate: 0.509090909090909
ROI: 0.0784909090909091


In [85]:
def deep_performance_analysis(weeks):
    """
    Analyze which types of bets perform best
    """
    all_results = []
    
    for week in weeks:
        df, summary = ev_betting(2025, week)
        df['week'] = week
        all_results.append(df)
    
    full_df = pd.concat(all_results)
    
    # Analysis by probability buckets
    full_df['prob_bucket'] = pd.cut(full_df['predicted_touchdown_probability'], 
                                      bins=[0, 0.30, 0.35, 0.40, 0.45,0.50,0.55,0.60],
                                      labels=['<30%', '30-35%', '35-40%', '40-45%', '45-50%', '50-55%', '>55%'])
    
    print("Hit Rate by Probability Bucket:")
    print(full_df.groupby('prob_bucket')['hit'].agg(['mean', 'count']))
    
    # Analysis by edge
    full_df['edge_bucket'] = pd.cut(full_df['model_edge'],
                                      bins=[-1, 0.05, 0.10, 0.15, 1.0],
                                      labels=['0-5%', '5-10%', '10-15%', '>15%'])
    
    print("\nROI by Edge Bucket:")
    print(full_df.groupby('edge_bucket').apply(
        lambda x: x['profit'].sum() / (len(x) * 10)
    ))
    
    return full_df

In [86]:
from itertools import combinations
import pandas as pd
import numpy as np

def calculate_parlay_payout(odds_list, stake=10):
    """
    Calculate parlay payout from American odds
    
    Args:
        odds_list: List of American odds (e.g., [150, -110, 200])
        stake: Bet amount
    
    Returns:
        total_payout (includes stake), profit
    """
    # Convert American odds to decimal multipliers
    decimal_odds = []
    for odds in odds_list:
        if odds > 0:
            decimal = 1 + (odds / 100.0)
        else:
            decimal = 1 + (100.0 / abs(odds))
        decimal_odds.append(decimal)
    
    # Multiply all decimal odds together
    total_multiplier = np.prod(decimal_odds)
    total_payout = stake * total_multiplier
    profit = total_payout - stake
    
    return total_payout, profit


def generate_round_robin_combinations(bets_df, parlay_sizes=[2, 3]):
    """
    Generate all round robin parlay combinations
    
    Args:
        bets_df: DataFrame with bet info (player_id, price, etc.)
        parlay_sizes: List of parlay sizes to create (e.g., [2, 3] for 2-leg and 3-leg)
    
    Returns:
        List of parlay combinations (each is a list of indices)
    """
    n_bets = len(bets_df)
    all_parlays = []
    
    for size in parlay_sizes:
        if size <= n_bets:
            # Generate all combinations of that size
            combos = list(combinations(range(n_bets), size))
            all_parlays.extend(combos)
    
    return all_parlays


def simulate_round_robin(bets_df, scorers, parlay_sizes=[2, 3], stake_per_parlay=10):
    """
    Simulate round robin betting strategy
    
    Args:
        bets_df: DataFrame with your weekly picks
        scorers: DataFrame with actual TD scorers
        parlay_sizes: Sizes of parlays to include (e.g., [2, 3])
        stake_per_parlay: Amount to bet on each parlay
    
    Returns:
        results_df, summary
    """
    # Merge to determine hits
    bets = bets_df.copy()
    bets = bets.merge(
        scorers[['player_id', 'tds']], on='player_id', how='left'
    )
    bets['tds'] = bets['tds'].fillna(0).astype(int)
    bets['hit'] = bets['tds'] > 0
    
    # Generate all parlay combinations
    parlay_combos = generate_round_robin_combinations(bets, parlay_sizes)
    
    # Simulate each parlay
    parlay_results = []
    
    for combo in parlay_combos:
        # Get bets in this parlay
        parlay_bets = bets.iloc[list(combo)]
        
        # Check if parlay wins (all legs must hit)
        parlay_wins = parlay_bets['hit'].all()
        
        # Calculate payout
        if parlay_wins:
            total_payout, profit = calculate_parlay_payout(
                parlay_bets['price'].tolist(), 
                stake_per_parlay
            )
        else:
            total_payout = 0
            profit = -stake_per_parlay
        
        parlay_results.append({
            'parlay_size': len(combo),
            'players': ', '.join(parlay_bets['player_display_name'].tolist()),
            'odds': parlay_bets['price'].tolist(),
            'all_hit': parlay_wins,
            'stake': stake_per_parlay,
            'payout': total_payout,
            'profit': profit
        })
    
    results_df = pd.DataFrame(parlay_results)
    
    # Calculate summary
    total_stake = len(parlay_combos) * stake_per_parlay
    total_profit = results_df['profit'].sum()
    winning_parlays = results_df['all_hit'].sum()
    
    summary = {
        'total_parlays': len(parlay_combos),
        'winning_parlays': int(winning_parlays),
        'win_rate': round(winning_parlays / len(parlay_combos), 3),
        'total_stake': total_stake,
        'total_payout': round(results_df['payout'].sum(), 2),
        'total_profit': round(total_profit, 2),
        'roi': round(total_profit / total_stake, 3)
    }
    
    return results_df, summary


def compare_betting_strategies(bets_df, scorers, stake=10):
    """
    Compare straight bets vs round robin strategies
    """
    print("="*70)
    print("BETTING STRATEGY COMPARISON")
    print("="*70)
    
    # Strategy 1: Straight Bets (current approach)
    print("\n📊 STRATEGY 1: STRAIGHT BETS (Current)")
    print("-"*70)
    straight_results, straight_summary = simulate_betting(bets_df, scorers)
    display(straight_summary)
    
    # Strategy 2: Round Robin 2-leg parlays
    print("\n📊 STRATEGY 2: ROUND ROBIN (2-Leg Parlays)")
    print("-"*70)
    rr2_results, rr2_summary = simulate_round_robin(bets_df, scorers, parlay_sizes=[2], stake_per_parlay=stake)
    display(rr2_summary)
    
    # Strategy 3: Round Robin 3-leg parlays
    print("\n📊 STRATEGY 3: ROUND ROBIN (3-Leg Parlays)")
    print("-"*70)
    rr3_results, rr3_summary = simulate_round_robin(bets_df, scorers, parlay_sizes=[3], stake_per_parlay=stake)
    display(rr3_summary)
    
    # Strategy 4: Full Round Robin (2-leg + 3-leg)
    print("\n📊 STRATEGY 4: FULL ROUND ROBIN (2-Leg + 3-Leg)")
    print("-"*70)
    full_rr_results, full_rr_summary = simulate_round_robin(bets_df, scorers, parlay_sizes=[2, 3], stake_per_parlay=stake)
    display(full_rr_summary)
    
    # Comparison table
    print("\n📈 STRATEGY COMPARISON TABLE")
    print("-"*70)
    comparison = pd.DataFrame([
        {
            'Strategy': 'Straight Bets',
            'Total Stake': straight_summary['bets'] * stake,
            'Hit Rate': straight_summary['hit_rate'],
            'Total Profit': straight_summary['total_profit'],
            'ROI': straight_summary['roi']
        },
        {
            'Strategy': 'RR 2-Leg',
            'Total Stake': rr2_summary['total_stake'],
            'Hit Rate': rr2_summary['win_rate'],
            'Total Profit': rr2_summary['total_profit'],
            'ROI': rr2_summary['roi']
        },
        {
            'Strategy': 'RR 3-Leg',
            'Total Stake': rr3_summary['total_stake'],
            'Hit Rate': rr3_summary['win_rate'],
            'Total Profit': rr3_summary['total_profit'],
            'ROI': rr3_summary['roi']
        },
        {
            'Strategy': 'Full RR (2+3)',
            'Total Stake': full_rr_summary['total_stake'],
            'Hit Rate': full_rr_summary['win_rate'],
            'Total Profit': full_rr_summary['total_profit'],
            'ROI': full_rr_summary['roi']
        }
    ])
    
    display(comparison)
    
    return {
        'straight': (straight_results, straight_summary),
        'rr2': (rr2_results, rr2_summary),
        'rr3': (rr3_results, rr3_summary),
        'full_rr': (full_rr_results, full_rr_summary)
    }


def analyze_round_robin_by_hit_count(season, week, parlay_size=2):
    """
    Analyze round robin profitability based on how many picks hit
    """
    scorers = get_weekly_scorers(season, week)
    predictions = pd.read_csv(f'predictions/predictions_week_{week}.csv')
    
    # Get top 5 WR picks
    top_wr = predictions[predictions['position'] == 'WR'].sort_values(
        by='predicted_touchdown_probability', ascending=False
    ).head(5)
    
    # Determine how many hit
    top_wr = top_wr.merge(scorers[['player_id', 'tds']], on='player_id', how='left')
    top_wr['tds'] = top_wr['tds'].fillna(0).astype(int)
    top_wr['hit'] = top_wr['tds'] > 0
    
    hits = top_wr['hit'].sum()
    
    # Simulate round robin
    _, rr_summary = simulate_round_robin(top_wr, scorers, parlay_sizes=[parlay_size])
    
    print(f"Week {week}: {hits}/5 picks hit")
    print(f"Round Robin {parlay_size}-Leg Profit: ${rr_summary['total_profit']}")
    print(f"Round Robin {parlay_size}-Leg ROI: {rr_summary['roi']:.1%}")
    print()
    
    return hits, rr_summary


def season_round_robin_analysis(season, weeks, parlay_sizes=[2, 3]):
    """
    Full season analysis of round robin performance
    """
    print("="*70)
    print(f"ROUND ROBIN SEASON ANALYSIS - {season}")
    print("="*70)
    
    all_weeks_data = []
    
    for week in weeks:
        scorers = get_weekly_scorers(season, week)
        predictions = pd.read_csv(f'predictions/predictions_week_{week}.csv')
        
        top_wr = predictions[predictions['position'] == 'WR'].sort_values(
            by='predicted_touchdown_probability', ascending=False
        ).head(5)
        
        # Get results
        strategies = compare_betting_strategies(top_wr, scorers)
        
        all_weeks_data.append({
            'week': week,
            'straight_roi': strategies['straight'][1]['roi'],
            'rr2_roi': strategies['rr2'][1]['roi'],
            'rr3_roi': strategies['rr3'][1]['roi'],
            'full_rr_roi': strategies['full_rr'][1]['roi'],
            'straight_profit': strategies['straight'][1]['total_profit'],
            'rr2_profit': strategies['rr2'][1]['total_profit'],
            'rr3_profit': strategies['rr3'][1]['total_profit'],
            'full_rr_profit': strategies['full_rr'][1]['total_profit']
        })
    
    results_df = pd.DataFrame(all_weeks_data)
    
    print("\n📊 WEEKLY RESULTS")
    print(results_df.to_string(index=False))
    
    print("\n📈 SEASON TOTALS")
    totals = pd.DataFrame([{
        'Strategy': 'Straight Bets',
        'Total Profit': results_df['straight_profit'].sum(),
        'Avg Weekly ROI': results_df['straight_roi'].mean()
    }, {
        'Strategy': 'RR 2-Leg',
        'Total Profit': results_df['rr2_profit'].sum(),
        'Avg Weekly ROI': results_df['rr2_roi'].mean()
    }, {
        'Strategy': 'RR 3-Leg',
        'Total Profit': results_df['rr3_profit'].sum(),
        'Avg Weekly ROI': results_df['rr3_roi'].mean()
    }, {
        'Strategy': 'Full RR (2+3)',
        'Total Profit': results_df['full_rr_profit'].sum(),
        'Avg Weekly ROI': results_df['full_rr_roi'].mean()
    }])
    
    display(totals)
    
    return results_df, totals

In [87]:
# Cell: Full Season Round Robin Analysis
#season_results, season_totals = season_round_robin_analysis(2025, range(1, 8))

In [88]:
#deep_performance_analysis(range(1, 8))

In [89]:
current_week = 9


rf_predictions = pd.read_csv(f'predictions/predictions_week_{current_week}.csv')

rf_predictions = rf_predictions[['player_id', 'player_display_name', 'position','team','opponent_team', 'predicted_touchdown_probability', 'price', 'model_edge','market_implied_prob']]
rf_predictions = rf_predictions.sort_values(by='predicted_touchdown_probability', ascending=False)


In [90]:
#rf_predictions = get_weekly_results(2025, current_week)

In [91]:
wr_plays = rf_predictions.head(20)
wr_plays

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
0,00-0038543,Jaxon Smith-Njigba,WR,SEA,WAS,0.533415,-115.0,-0.001469,0.534884
1,00-0039075,Puka Nacua,WR,LA,NO,0.486660,-125.0,-0.068896,0.555556
2,00-0036963,Amon-Ra St. Brown,WR,DET,MIN,0.472284,-115.0,-0.062600,0.534884
3,00-0038996,Tucker Kraft,TE,GB,CAR,0.463870,100.0,-0.036130,0.500000
4,00-0036252,Michael Pittman,WR,IND,PIT,0.451082,185.0,0.100205,0.350877
5,00-0037247,George Pickens,WR,DAL,ARI,0.446119,130.0,0.011336,0.434783
6,00-0031381,Davante Adams,WR,LA,NO,0.435680,-110.0,-0.088129,0.523810
7,00-0036358,CeeDee Lamb,WR,DAL,ARI,0.429319,105.0,-0.058486,0.487805
8,00-0037744,Trey McBride,TE,ARI,DAL,0.428115,105.0,-0.059690,0.487805
9,00-0039915,Ladd McConkey,WR,LAC,TEN,0.417576,160.0,0.032961,0.384615


In [92]:
ev_wr = rf_predictions.sort_values(by='model_edge', ascending=False)
ev_wr = ev_wr[ev_wr['price'] <= 400].head(15)
ev_wr

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
4,00-0036252,Michael Pittman,WR,IND,PIT,0.451082,185.0,0.100205,0.350877
25,00-0037239,Chris Olave,WR,NO,LA,0.308353,330.0,0.075794,0.232558
14,00-0038544,Quentin Johnston,WR,LAC,TEN,0.369483,215.0,0.052023,0.317460
29,00-0037664,Alec Pierce,WR,IND,PIT,0.292854,300.0,0.042854,0.250000
9,00-0039915,Ladd McConkey,WR,LAC,TEN,0.417576,160.0,0.032961,0.384615
34,00-0038997,Josh Downs,WR,IND,PIT,0.275839,310.0,0.031936,0.243902
46,00-0033307,Kendrick Bourne,WR,SF,NYG,0.235508,360.0,0.018116,0.217391
5,00-0037247,George Pickens,WR,DAL,ARI,0.446119,130.0,0.011336,0.434783
27,00-0038994,Jordan Addison,WR,MIN,DET,0.303762,235.0,0.005255,0.298507
28,00-0040124,Tetairoa McMillan,WR,CAR,GB,0.293020,240.0,-0.001098,0.294118


In [93]:
ev = rf_predictions[rf_predictions['model_edge'] >= 0.05]
ev = ev[ev['model_edge'] <= 0.10]
ev = ev[ev['price'] <= 400]
ev

,player_id,player_display_name,position,team,opponent_team,predicted_touchdown_probability,price,model_edge,market_implied_prob
14,00-0038544,Quentin Johnston,WR,LAC,TEN,0.369483,215.0,0.052023,0.317460
25,00-0037239,Chris Olave,WR,NO,LA,0.308353,330.0,0.075794,0.232558


In [94]:
predictions = pd.read_csv(f'predictions/predictions_week_{current_week}.csv')
predictions = predictions.groupby('opponent_team')['passing_tds_allowed_to_WR'].mean()
predictions = predictions.sort_values(ascending=False)
predictions



opponent_team
JAX    1.906749
DAL    1.861060
DET    1.650092
MIN    1.434915
LV     1.430732
SF     1.205700
GB     1.118051
CHI    1.078828
TEN    1.060533
PIT    1.018725
IND    0.996420
ATL    0.965704
SEA    0.963779
WAS    0.895258
NYG    0.876382
CAR    0.768706
NO     0.748867
LAC    0.733379
KC     0.716572
LA     0.686317
NE     0.629367
BUF    0.627521
CIN    0.593313
MIA    0.534524
DEN    0.403467
ARI    0.379979
HOU    0.376976
Name: passing_tds_allowed_to_WR, dtype: float64

In [95]:
import nflreadpy as nfl

exp = nfl.load_ff_opportunity(seasons = [2020, 2021, 2022, 2023, 2024, 2025], stat_type='weekly')
exp.to_pandas()

exp




season,posteam,week,game_id,player_id,full_name,position,pass_attempt,rec_attempt,rush_attempt,pass_air_yards,rec_air_yards,pass_completions,receptions,pass_completions_exp,receptions_exp,pass_yards_gained,rec_yards_gained,rush_yards_gained,pass_yards_gained_exp,rec_yards_gained_exp,rush_yards_gained_exp,pass_touchdown,rec_touchdown,rush_touchdown,pass_touchdown_exp,rec_touchdown_exp,rush_touchdown_exp,pass_two_point_conv,rec_two_point_conv,rush_two_point_conv,pass_two_point_conv_exp,rec_two_point_conv_exp,rush_two_point_conv_exp,pass_first_down,rec_first_down,rush_first_down,…,pass_fantasy_points_exp_team,rec_fantasy_points_exp_team,rush_fantasy_points_exp_team,pass_fantasy_points_team,rec_fantasy_points_team,rush_fantasy_points_team,pass_completions_diff_team,receptions_diff_team,pass_yards_gained_diff_team,rec_yards_gained_diff_team,rush_yards_gained_diff_team,pass_touchdown_diff_team,rec_touchdown_diff_team,rush_touchdown_diff_team,pass_two_point_conv_diff_team,rec_two_point_conv_diff_team,rush_two_point_conv_diff_team,pass_first_down_diff_team,rec_first_down_diff_team,rush_first_down_diff_team,pass_interception_diff_team,rec_interception_diff_team,pass_fantasy_points_diff_team,rec_fantasy_points_diff_team,rush_fantasy_points_diff_team,total_yards_gained_team,total_yards_gained_exp_team,total_yards_gained_diff_team,total_touchdown_team,total_touchdown_exp_team,total_touchdown_diff_team,total_first_down_team,total_first_down_exp_team,total_first_down_diff_team,total_fantasy_points_team,total_fantasy_points_exp_team,total_fantasy_points_diff_team
str,str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2020""","""SF""",1.0,"""2020_01_ARI_SF""","""00-0031345""","""Jimmy Garoppolo""","""QB""",33.0,0.0,1.0,220.0,0.0,19.0,0.0,22.24,0.0,259.0,0.0,9.0,228.15,0.0,7.04,2.0,0.0,0.0,2.24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,…,16.94,58.52,17.19,18.36,56.9,12.3,-3.24,-3.24,30.85,30.84,6.59,-0.24,-0.25,-0.93,0.0,0.0,0.0,-1.29,-1.31,-1.08,-0.58,-0.59,1.42,-1.62,-4.89,382.0,344.57,37.43,2.0,3.18,-1.18,16.0,18.39,-2.39,69.2,75.71,-6.51
"""2020""","""SF""",1.0,"""2020_01_ARI_SF""","""00-0033288""","""George Kittle""","""TE""",0.0,5.0,1.0,0.0,26.0,0.0,4.0,0.0,3.93,0.0,44.0,9.0,0.0,38.61,5.87,0.0,0.0,0.0,0.0,0.23,0.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,…,16.94,58.52,17.19,18.36,56.9,12.3,-3.24,-3.24,30.85,30.84,6.59,-0.24,-0.25,-0.93,0.0,0.0,0.0,-1.29,-1.31,-1.08,-0.58,-0.59,1.42,-1.62,-4.89,382.0,344.57,37.43,2.0,3.18,-1.18,16.0,18.39,-2.39,69.2,75.71,-6.51
"""2020""","""ARI""",1.0,"""2020_01_ARI_SF""","""00-0035228""","""Kyler Murray""","""QB""",40.0,0.0,13.0,192.0,0.0,26.0,0.0,28.67,0.0,230.0,0.0,91.0,270.24,0.0,75.43,1.0,0.0,1.0,0.47,0.0,0.07,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,4.0,…,11.56,58.51,21.89,11.2,55.0,30.0,-2.67,-2.68,-40.24,-40.26,4.97,0.53,0.54,1.27,0.0,0.0,0.0,0.42,0.42,3.07,0.44,0.43,-0.36,-3.51,8.11,410.0,445.29,-35.29,3.0,1.19,1.81,25.0,21.51,3.49,85.0,80.4,4.6
"""2020""","""ARI""",1.0,"""2020_01_ARI_SF""","""00-0030564""","""DeAndre Hopkins""","""WR""",0.0,16.0,0.0,0.0,103.0,0.0,14.0,0.0,11.56,0.0,151.0,0.0,0.0,118.96,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,…,11.56,58.51,21.89,11.2,55.0,30.0,-2.67,-2.68,-40.24,-40.26,4.97,0.53,0.54,1.27,0.0,0.0,0.0,0.42,0.42,3.07,0.44,0.43,-0.36,-3.51,8.11,410.0,445.29,-35.29,3.0,1.19,1.81,25.0,21.51,3.49,85.0,80.4,4.6
"""2020""","""ARI""",1.0,"""2020_01_ARI_SF""","""00-0022921""","""Larry Fitzgerald""","""WR""",0.0,5.0,0.0,0.0,15.0,0.0,4.0,0.0,3.88,0.0,34.0,0.0,0.0,32.6,0.0,0.0,0.0,0.0,0.0,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,…,11.56,58.51,21.89,11.2,55.0,30.0,-2.67,-2.68,-40.24,-40.26,4.97,0.53,0.54,1.27,0.0,0.0,0.0,0.42,0.42,3.07,0.44,0.43,-0.36,-3.51,8.11,410.0,445.29,-35.29,3.0,1.19,1.81,25.0,21.51,3.49,85.0